<a href="https://colab.research.google.com/github/ViggenKorir/q3-ai-Developers-Mania/blob/Sprint26/Sprint26_S6_MCP_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup**

Install `fastmcp`, the standard Python framework for building and testing MCP servers and clients.

In [ ]:
# Cell 1: Install FastMCP
!pip install -q fastmcp nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.0/862.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.5/159.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.1 MB/s eta 0:00:00


**Cell 2: Define the MCP Server (The "Kitchen")**

Here, we instantiate an MCP server representing a workplace support system. Notice how standard Python docstrings and type hints automatically turn into the natural language schema the AI reads.

In [ ]:
from fastmcp import FastMCP

# Initialize the MCP Server
server = FastMCP(
    name="CompanyKnowledgeAndTools",
    instructions="Provides company knowledge docs and administrative IT tools."
)

# Simulated in-memory database
EMPLOYEES_DB = {
    "alice": {"role": "Security Engineer", "status": "Active", "quota_gb": 50},
    "bob": {"role": "Data Analyst", "status": "Suspended", "quota_gb": 15},
    "david": {"role": "IT manager", "status": "Complicated", "quota_gb": 0},
}

# 1. MCP RESOURCE: Read-only background context (Passive)
@server.resource("company://security-policy")
def get_security_policy() -> str:
    """Provides official company IT security guidelines."""
    return (
        "Security Policy 2026: All elevated actions on suspended accounts require approval. "
        "Default storage limit per user is 50 GB."
    )

# 2. MCP TOOL: An executable action with side effects (Active)
@server.tool
def inspect_account(username: str) -> dict:
    """Inspects an employee account details and current provisioning status."""
    user = EMPLOYEES_DB.get(username.lower())
    if not user:
        return {"error": f"User '{username}' not found."}
    return {"username": username, **user}

# 3. MCP TOOL: Modifying state
@server.tool
def update_storage_quota(username: str, additional_gb: int) -> dict:
    """Increases an employee's cloud storage quota in gigabytes."""
    user = EMPLOYEES_DB.get(username.lower())
    if not user:
        return {"success": False, "error": "User not found."}

    # Define allowed statuses for quota updates
    ALLOWED_STATUSES_FOR_QUOTA_UPDATE = ["Active"]

    # Reject action if user's status is not in the allowed list
    if user["status"] not in ALLOWED_STATUSES_FOR_QUOTA_UPDATE:
        return {"success": False, "error": f"Action rejected: Cannot modify accounts with status '{user['status']}'."}

    user["quota_gb"] += additional_gb
    return {"success": True, "new_quota_gb": user["quota_gb"]}

print("✅ MCP Server initialized with 1 Resource and 2 Tools!")

✅ MCP Server initialized with 1 Resource and 2 Tools!


**Cell 3: The MCP Client Discovery (The "Menu")**

In traditional APIs, client code must hardcode endpoints like GET /api/v1/user.
With MCP, the client connects to the server and runs dynamic discovery to learn what actions exist and their expected arguments.

In [ ]:
# Cell 3: Discover Server Capabilities
import nest_asyncio
import asyncio
from fastmcp import Client

nest_asyncio.apply()  # Enables asyncio loop in Colab

async def explore_mcp():
    # FastMCP connects to in-memory server instances directly
    async with Client(server) as client:
        print("--- 1. DISCOVERED RESOURCES (Passive Context) ---")
        resources = await client.list_resources()
        for res in resources:
            print(f"URI: {res.uri}")
            print(f"Description: {res.description}\n")

        print("--- 2. DISCOVERED TOOLS (Callable Actions) ---")
        tools = await client.list_tools()
        for tool in tools:
            print(f"Tool Name: {tool.name}")
            print(f"Description: {tool.description}")
            print(f"Parameters Schema: {tool.inputSchema['properties']}\n")

asyncio.run(explore_mcp())

--- 1. DISCOVERED RESOURCES (Passive Context) ---
URI: company://security-policy
Description: Provides official company IT security guidelines.

--- 2. DISCOVERED TOOLS (Callable Actions) ---
Tool Name: inspect_account
Description: Inspects an employee account details and current provisioning status.
Parameters Schema: {'username': {'type': 'string'}}

Tool Name: update_storage_quota
Description: Increases an employee's cloud storage quota in gigabytes.
Parameters Schema: {'username': {'type': 'string'}, 'additional_gb': {'type': 'integer'}}



**Cell 4: Simulating the AI Agent Loop**

This script simulates an AI agent receiving a user prompt, picking the correct MCP primitive based on its self-describing metadata, and handling errors cleanly.

In [ ]:
# Cell 4: Execute Client Requests
async def run_agent_workflow():
    async with Client(server) as client:
        print("🤖 [Step 1: Reading Background Context]")
        policy_doc = await client.read_resource("company://security-policy")
        print(f"Policy Context loaded: '{policy_doc[0].text[:62]}...'\n")

        print("🤖 [Step 2: Checking Alice's Profile]")
        # Client invokes the tool discovered in Step 3
        alice_info = await client.call_tool("inspect_account", {"username": "alice"})
        print(f"Tool Response: {alice_info.data}\n")

        print("🤖 [Step 3: Attempting Action on Bob (Suspended)]")
        # Demonstrating controlled error handling over the standard protocol
        bob_action = await client.call_tool("update_storage_quota", {"username": "bob", "additional_gb": 20})
        print(f"Tool Response: {bob_action.data}\n")

        print("🤖 [Step 4: Attempting Action on David]")
        # Demonstrating controlled error handling over the standard protocol
        david_action = await client.call_tool("update_storage_quota", {"username": "david", "additional_gb": 20})
        print(f"Tool Response: {david_action.data}\n")

        # print("🤖 [Step 4: Executing Valid Action for Alice]")
        # quota_action = await client.call_tool("update_storage_quota", {"username": "alice", "additional_gb": 25})
        # print(f"Tool Response: {quota_action.data

        print("🤖 [Step 4: Executing Valid Action for Alice]")
        quota_action = await client.call_tool("update_storage_quota", {"username": "alice", "additional_gb": 25})
        print(f"Tool Response: {quota_action.data}")


asyncio.run(run_agent_workflow())

🤖 [Step 1: Reading Background Context]
Policy Context loaded: 'Security Policy 2026: All elevated actions on suspended accoun...'

🤖 [Step 2: Checking Alice's Profile]
Tool Response: {'username': 'alice', 'role': 'Security Engineer', 'status': 'Active', 'quota_gb': 50}

🤖 [Step 3: Attempting Action on Bob (Suspended)]
Tool Response: {'success': False, 'error': 'Action rejected: Cannot modify suspended accounts.'}

🤖 [Step 4: Attempting Action on David]
Tool Response: {'success': True, 'new_quota_gb': 20}

🤖 [Step 4: Executing Valid Action for Alice]
Tool Response: {'success': True, 'new_quota_gb': 75}


**What Just Happened?**

*Self-Describing Interfaces:* The server didn't require OpenAPI YAML files or documentation pages. Python type annotations and docstrings became the schema that any MCP host automatically reads.

*Resource vs. Tool Separation:* Context was fetched safely with read_resource() (read-only), while data modifications were gated behind call_tool().

*No Direct Secret Sharing: *The consumer simply calls the tool; connection details, permission gates, and database modifications stay isolated inside the MCP server.